# Day 9 — ILT 2: Watermark-Based Incremental Loading

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 1 — Need for Incremental Loading |
| **Duration** | 90 minutes |
| **Format** | Instructor-led — every query below is real and safe to run against `gbmart` (GlobalMart's actual run; your own catalog will be named differently) |

### Learning Objectives
- Explain exactly how a cursor/watermark-based connector decides what's "new"
- Trace GlobalMart's real `orders_data_ingestion_cdc` pipeline configuration end-to-end
- Verify an incremental pipeline run using real Bronze-layer checks

---

## The Mechanism, in One Query

Every watermark-based incremental load boils down to one query shape, run on a schedule:

```sql
SELECT * FROM source_table WHERE updated_at > :last_watermark_value;
-- ... then, only after that batch is successfully written downstream:
UPDATE control_table SET last_watermark_value = MAX(updated_at) FROM this_batch;
```

Two things have to be true for this to work correctly:
1. **The source column must be reliably bumped on every change.** GlobalMart's Postgres `orders` table has a trigger, `trg_orders_updated_at`, that sets `updated_at = now()` automatically on every `UPDATE` — nobody has to remember to do it by hand.
2. **The watermark must only advance after a successful write.** If you advance the watermark before confirming the downstream write succeeded, a failed run silently loses that batch forever.

## GlobalMart's Real Pipeline Configuration

This is the actual, live Lakeflow Connect ingestion pipeline spec for `orders_data_ingestion_cdc` (built in Day 2) — not a hypothetical example:

```json
{
  "name": "orders_data_ingestion_cdc",
  "spec": {
    "catalog": "gbmart", "schema": "bronze", "serverless": true,
    "ingestion_definition": {
      "connection_name": "ecom_gbmart_conn",
      "source_type": "POSTGRESQL",
      "objects": [
        { "table": {
            "source_catalog": "postgres", "source_schema": "globalmart", "source_table": "orders",
            "destination_catalog": "gbmart", "destination_schema": "bronze", "destination_table": "orders",
            "table_configuration": {
              "primary_keys": ["orderid"],
              "query_based_connector_config": { "cursor_columns": ["updated_at"] }
        }}},
        { "table": {
            "source_catalog": "postgres", "source_schema": "globalmart", "source_table": "order_items",
            "destination_catalog": "gbmart", "destination_schema": "bronze", "destination_table": "order_items",
            "table_configuration": {
              "primary_keys": ["orderitemid"],
              "query_based_connector_config": { "cursor_columns": ["updated_at"] }
        }}}
      ]
    }
  }
}
```

Point out to the class: `primary_keys: ["orderid"]`, not `order_id`. Bronze columns coming from this Postgres source keep the source's own naming (`orderid`, `customerid`, `orderchannel`, `shippingdate`, `actualdeliverydate`) — no underscores. Day 5's Silver layer is what renames these to `order_id`, `customer_id`, etc. Bronze is raw-as-received, always.

In [ ]:
# Live, read-only. Confirms the real column naming discussed above — run this and
# actually look at the column names before moving on, don't just take the slide's word for it.
spark.sql("DESCRIBE gbmart.bronze.orders").select("col_name", "data_type").show(20, truncate=False)

## Verifying an Incremental Run Actually Happened

How do you *prove* the cursor picked up new changes — instead of just trusting the pipeline UI's green "Succeeded"? You run queries and actually look at the rows.

This is the exact six-query pattern used against GlobalMart's own `orders_data_ingestion_cdc` pipeline, the last time it ran after three real changes landed in the Postgres source:

1. an **UPDATE** to order `OR-000478`
2. two brand-new **INSERT**s — orders `OR-900001` and `OR-900002`

Harsh Kumar's rule of thumb: never trust a pipeline run just because the UI says "Succeeded" — go look at the rows it was supposed to bring in. That's all six checks below are: going and looking.

### Check 1 — Did the Update on `OR-000478` Come Through?

Somewhere in Postgres, order `OR-000478` was updated — its delivery finally got confirmed. Heemansh Bhawsar wants to know: did the cursor actually pick that up on the last run, or is Bronze still showing stale data?

Expect: `actualdeliverydate` populated, and `updated_at` bumped to a recent timestamp. Remember, the `trg_orders_updated_at` trigger sets that automatically on every Postgres `UPDATE` — so a fresh `updated_at` here is proof the row was truly touched, not just proof someone re-ran a query.

In [ ]:
# Check 1 — real, live, read-only. Same query GlobalMart's own pipeline verification runs.
spark.sql("""
    SELECT orderid, customerid, shippingdate, actualdeliverydate, orderchannel, updated_at
    FROM gbmart.bronze.orders
    WHERE orderid = 'OR-000478'
""").display()

### Check 2 — Did the 2 New Orders Land?

`OR-900001` and `OR-900002` were inserted straight into the Postgres `orders` table. Devanand P checks: does the cursor treat a brand-new row the same way it treats an updated one? It should — both show up for the same reason, because both have an `updated_at` newer than the last watermark. The cursor doesn't distinguish "new row" from "changed row"; it only knows "row with a newer timestamp than I've seen before."

Expect 2 rows back.

In [ ]:
# Check 2 — real, live, read-only.
spark.sql("""
    SELECT orderid, customerid, orderdate, orderchannel, updated_at
    FROM gbmart.bronze.orders
    WHERE orderid IN ('OR-900001', 'OR-900002')
""").display()

### Check 3 — Total Row Count on `bronze.orders`

At the time this pipeline's verification was written, the author's own notes said the total should land at `126,036 + 2 = 126,038` — the pre-existing count plus the two new inserts from Check 2.

**Treat that number as a snapshot, not a fact.** GlobalMart's real `orders` table keeps growing every day this pipeline runs. The only way to know the *current* true count is to run `COUNT(*)` live, right now — never hardcode a row count into a check and trust it forever. Kumar Pratik always re-runs this one live before trusting any older number written down in a doc.

In [ ]:
# Check 3 — live COUNT(*), not a hardcoded number. Compare this against whatever
# baseline you noted down before the pipeline's last run.
spark.sql("SELECT COUNT(*) AS total_orders FROM gbmart.bronze.orders").display()

### Check 4 — Did Matching `order_items` Land?

Only relevant if the two new orders above also had matching line items inserted into Postgres `order_items`. If that optional insert step was run, expect 2 rows. If it wasn't, **0 rows is the correct result here, not a failure** — it just means `order_items` never had anything to bring in for these two orders.

In [ ]:
# Check 4 — real, live, read-only. 0 rows is a valid, correct outcome here, not a bug.
spark.sql("""
    SELECT orderitemid, orderid, productid, quantity
    FROM gbmart.bronze.order_items
    WHERE orderid IN ('OR-900001', 'OR-900002')
""").display()

### Check 5 — Total Row Count on `bronze.order_items`

Same caveat as Check 3. The author's notes say `377,866` if the `order_items` insert wasn't run, or `377,868` if it was. Same rule applies: this is the baseline at the time it was written down, not a permanent number. Run the count live and compare against your own most recent baseline — not against whatever a slide from months ago says.

In [ ]:
# Check 5 — live COUNT(*).
spark.sql("SELECT COUNT(*) AS total_order_items FROM gbmart.bronze.order_items").display()

### Check 6 — Combined Glance

One query, both counts side by side — the quickest way to eyeball both tables in a single glance before digging into individual rows.

In [ ]:
# Check 6 — real, live, read-only.
spark.sql("""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM gbmart.bronze.orders
    UNION ALL
    SELECT 'order_items', COUNT(*) FROM gbmart.bronze.order_items
""").display()

### Bonus — Beyond the Real Six-Check Pattern

The six checks above are the exact pattern used against GlobalMart's real pipeline. Two more checks aren't part of that original six, but they're worth knowing — they answer a slightly different question: *when* did the pipeline last actually change something, and *which* rows moved most recently?

In [ ]:
# Bonus A — DESCRIBE HISTORY. A new version appears only when the cursor found
# something newer than its last watermark. If you re-trigger the pipeline and no new
# version shows up, that's expected when nothing changed in Postgres since the last run.
spark.sql("DESCRIBE HISTORY gbmart.bronze.orders") \
    .select("version", "timestamp", "operation") \
    .orderBy("version", ascending=False) \
    .show(5, truncate=False)

# Bonus B — the 10 most recently updated rows. Since the cursor column is updated_at,
# these are exactly the rows the most recent pipeline run would have picked up.
spark.sql("""
    SELECT orderid, customerid, orderchannel, updated_at
    FROM gbmart.bronze.orders
    ORDER BY updated_at DESC
    LIMIT 10
""").display()

## Proving the Blind Spot (Carried Forward from Day 2)

If a row were hard-deleted from `postgres.globalmart.orders` right now, the query above (`WHERE updated_at > last_watermark`) would simply never see it — there's no row left to have a newer `updated_at`. The row would sit in `gbmart.bronze.orders` forever, undeleted, with no error, no warning, and no signal that anything is wrong. This is exactly what Day 2's HOL had you prove hands-on by deleting a test row and watching it not sync.

**This is the one gap CDF closes** — which is exactly why GlobalMart uses CDF (not cursor-based watermarking) for the Bronze → Silver hop, covered next in ILT 3.